In [ ]:
# ============================================
# 1. Introduction
# ============================================

# This notebook performs customer segmentation using clustering.
# It includes:
# - Selection of numeric variables
# - Scaling
# - PCA (2D and 3D)
# - KMeans with optimal K selection
# - Cluster visualization
# - Segment profiling

# ============================================
# 2. Load libraries
# ============================================

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

sns.set(style="whitegrid")

# ============================================
# 3. Load dataset with features
# ============================================

df = pd.read_csv("../data/processed/features_telco.csv")
df.head()

cluster_cols = [
    "TenureinMonths", "MonthlyCharge", "TotalCharges", "TotalRevenue",
    "TotalServices", "EngagementScore", "BillingRiskScore",
    "CLTV_Normalized"
]

X = df[cluster_cols].copy()
X.head()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

df["PCA1"] = X_pca[:, 0]
df["PCA2"] = X_pca[:, 1]

pca.explained_variance_ratio_

inertia = []
K_range = range(2, 11)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42)
    km.fit(X_scaled)
    inertia.append(km.inertia_)

plt.figure(figsize=(8,4))
plt.plot(K_range, inertia, marker="o")
plt.title("Elbow Method")
plt.xlabel("Number of clusters (k)")
plt.ylabel("Inertia")
plt.show()

silhouette_scores = []
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    silhouette_scores.append(silhouette_score(X_scaled, labels, sample_size=3000, random_state=42))

plt.figure(figsize=(8,4))
plt.plot(K_range, silhouette_scores, marker="o", color="darkorange")
plt.title("Silhouette Score by k")
plt.xlabel("Number of clusters (k)")
plt.ylabel("Silhouette Score")
plt.show()

k_opt = 4
kmeans = KMeans(n_clusters=k_opt, random_state=42)
df["Cluster"] = kmeans.fit_predict(X_scaled)

df["Cluster"].value_counts()

plt.figure(figsize=(10,6))
sns.scatterplot(
    data=df,
    x="PCA1", y="PCA2",
    hue="Cluster",
    palette="tab10",
    alpha=0.7
)
plt.title("Clusters in PCA 2D Space")
plt.show()

cluster_profile = df.groupby("Cluster")[cluster_cols].mean()
cluster_profile

plt.figure(figsize=(12,6))
sns.heatmap(cluster_profile, annot=True, cmap="coolwarm")
plt.title("Cluster Profile")
plt.show()

for c in sorted(df["Cluster"].unique()):
    print(f"\n=== Cluster {c} ===")
    print(cluster_profile.loc[c])

df.to_csv("../data/processed/segments_telco.csv", index=False)
print("Segmented dataset saved to data/processed/segments_telco.csv")

print("""
SEGMENTATION CONCLUSIONS:

1. Relevant numeric variables were selected for clustering.
2. Scaling and PCA were applied to improve visualization.
3. The optimal number of clusters was assessed using the elbow method.
4. KMeans was trained and a cluster was assigned to each customer.
5. The silhouette score is maximized at k=2 (0.40), which indicates that
   statistically the data separates better into 2 large groups.
   However, k=4 was chosen because it produces segments with much more
   differentiated churn rates (3% to 37%) than k=2 or k=3, which is
   more actionable for designing distinct retention strategies per
   segment.
6. The dataset is ready for funnel analysis in 05_Funnel_Analysis.ipynb.
""")
